# YAGO3-10 — preprocessing

Prepares the YAGO3-10 inputs for the class-prediction experiments (vocabulary: see `GLOSSARY.md`).
Three outputs are written to `data/processed/yago3-10/`:

1. `train.tsv`, `valid.tsv`, `test.tsv`: the official **triple split**, unchanged (no header, same row order, no deduplication, no self-loop removal).
2. `entity_classes.tsv`: one row per (entity, **class**) pair, with the WordNet ancestors of every class added.
3. `entity_split.tsv`: the 80/20 **entity split** for every **experiment**, following the rules of the authors' `classification.ipynb` (Jain et al. 2021, pinned commit `2d93782`).

The acceptance checks (section 8) were written first and pin the behaviour; sections 1-7 produce the files they check.

In [1]:
from pathlib import Path
import csv
import hashlib
import math
import shutil
import urllib.request
import zipfile

import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

DATASET = 'yago3-10'
COMMITTED_TRAIN = 'yago3-10_train.csv'
candidates = [Path.cwd() / 'data', Path.cwd()]
DATA_DIR = next((p for p in candidates if (p / COMMITTED_TRAIN).is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError(f'{COMMITTED_TRAIN} not found. Run from the repository root or data/ directory.')
EXTERNAL_DIR = DATA_DIR / 'external' / DATASET
OUT_DIR = DATA_DIR / 'processed' / DATASET
OUT_DIR.mkdir(parents=True, exist_ok=True)

MIN_CLASS_SIZE = 40
TEST_SIZE = 0.2
RANDOM_STATE = 37
SPLITS = ['train', 'valid', 'test']

# Pinned sources.
HF = 'https://huggingface.co/datasets/VLyb/YAGO3-10/resolve/ecb04758ade513529e118c5c97bca4252a6bec67/'
AUTHORS = 'https://raw.githubusercontent.com/nitishajain/KGESemanticAnalysis/2d93782b55e169ec1f5e79c9c6245b709b44c86b/'
SOURCES = {  # file name -> (url, sha256)
    'train.csv': (HF + 'train.csv', '766918ef5d16eba8344eb99e604321ba04f9b92a3091f5f1874dc54b434920e0'),
    'valid.csv': (HF + 'valid.csv', 'c2b3eecf6d42a34d60149ee60ee1c443f40fc423c76e77198202750709c5431a'),
    'test.csv': (HF + 'test.csv', 'a66ca8982f74c61ad69465c12e5425b41e60d7c189ed54ab103906f769b60895'),
    'SimpleTypeFactsWordnetLevel.zip': (
        AUTHORS + 'data/yago/SimpleTypeFactsWordnetLevel.zip',
        'a7fd43b1c2af8b254c926b1e13ee37dc21265908e4674a1da67cef2372ced2ce'),
}
EXPECTED_TRIPLES = {'train': 1_079_040, 'valid': 5_000, 'test': 5_000}

# Experiments, verbatim from the authors' code (the one `level-3-Players` is normalised).
EXPERIMENTS = {
    'Level-1': ['wordnet_person_100007846', 'wordnet_organization_108008335',
                'wordnet_body_of_water_109225146', 'wordnet_product_104007894'],
    'Level-2-Organizations': ['wordnet_musical_organization_108246613', 'wordnet_party_108256968',
                              'wordnet_enterprise_108056231', 'wordnet_nongovernmental_organization_108009834'],
    'Level-2-Waterbodies': ['wordnet_stream_109448361', 'wordnet_lake_109328904', 'wordnet_ocean_109376198',
                            'wordnet_bay_109215664', 'wordnet_sea_109426788'],
    'Level-2-Persons': ['wordnet_artist_109812338', 'wordnet_officeholder_110371450', 'wordnet_writer_110794014',
                        'wordnet_scientist_110560637', 'wordnet_politician_110450303'],
    'Level-3-Writers': ['wordnet_journalist_110224578', 'wordnet_poet_110444194', 'wordnet_novelist_110363573',
                        'wordnet_scriptwriter_110564905', 'wordnet_dramatist_110030277',
                        'wordnet_essayist_110064405', 'wordnet_biographer_109855433'],
    'Level-3-Scientists': ['wordnet_social_scientist_110619642', 'wordnet_biologist_109855630',
                           'wordnet_physicist_110428004', 'wordnet_mathematician_110301261',
                           'wordnet_chemist_109913824', 'wordnet_linguist_110264437',
                           'wordnet_psychologist_110488865', 'wordnet_geologist_110127689',
                           'wordnet_computer_scientist_109951070', 'wordnet_research_worker_110523076'],
    'Level-3-Players': ['wordnet_football_player_110101634', 'wordnet_ballplayer_109835506',
                        'wordnet_soccer_player_110618342', 'wordnet_volleyball_player_110759047',
                        'wordnet_golfer_110136959'],
    'Level-3-Artists': ['wordnet_painter_110391653', 'wordnet_sculptor_110566072',
                        'wordnet_photographer_110426749', 'wordnet_illustrator_109812068',
                        'wordnet_printmaker_110475687'],
}
EXPERIMENT_CLASSES = sorted({c for classes in EXPERIMENTS.values() for c in classes})


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()


def read_triples_csv(path):
    # Entity names can contain commas, so use a real CSV parser (quoted fields).
    return pd.read_csv(path, dtype='string', keep_default_na=False)


def read_tsv(path, header=True):
    # QUOTE_NONE: the TSVs we write are never quoted, quote characters inside names are literal.
    return pd.read_csv(path, sep='\t', quoting=csv.QUOTE_NONE, dtype='string', keep_default_na=False,
                       encoding='utf-8', header=0 if header else None)


def write_tsv(df, path, header=True):
    # Written by hand: UTF-8, LF line endings, never quoted. Values must not contain control separators.
    for col in df.columns:
        assert not df[col].str.contains('[\t\n\r]', regex=True).any(), f'tab/newline inside column {col!r}'
    lines = df.iloc[:, 0]
    for col in df.columns[1:]:
        lines = lines + '\t' + df[col]
    text = lines.str.cat(sep='\n') + '\n'
    if header:
        text = '\t'.join(df.columns) + '\n' + text
    with open(path, 'w', encoding='utf-8', newline='\n') as f:
        f.write(text)

## 1. Download and verify

Sources are pinned to commits and their SHA256 is asserted: the triple split from Hugging Face and the class labels (YAGO `rdf:type` facts restricted to WordNet classes) from the authors' repository. Downloads go to `data/external/yago3-10/` (gitignored).

In [2]:
def fetch(name, url, expected):
    dest = EXTERNAL_DIR / name
    if not (dest.is_file() and sha256(dest) == expected):
        dest.parent.mkdir(parents=True, exist_ok=True)
        request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(request) as response, open(dest, 'wb') as f:
            shutil.copyfileobj(response, f)
    assert sha256(dest) == expected, f'SHA256 mismatch for {name}'
    return dest


downloads = {name: fetch(name, url, sha) for name, (url, sha) in SOURCES.items()}
pd.Series({name: f'{path.stat().st_size:,} bytes' for name, path in downloads.items()}, name='downloaded')

train.csv                          51,310,750 bytes
valid.csv                             236,899 bytes
test.csv                              238,601 bytes
SimpleTypeFactsWordnetLevel.zip     6,215,441 bytes
Name: downloaded, dtype: str

## 2. Triple split

The official split is copied unchanged: no deduplication, no self-loop removal, same row order. Only the format changes (quoted CSV with header to unquoted TSV without header). The downloaded train split must hold the same triples as the train file committed to this repository (compared by content, since `core.autocrlf` makes working-tree bytes differ). The single exception to "unchanged": the first row of the source `test.csv` has a stray U+FEFF glued to its head entity, which is removed (see the cell output).

In [3]:
triples = {split: read_triples_csv(EXTERNAL_DIR / f'{split}.csv') for split in SPLITS}
for split, df in triples.items():
    assert list(df.columns) == ['head', 'relation', 'tail'], (split, list(df.columns))
    assert len(df) == EXPECTED_TRIPLES[split], (split, len(df))

# The only deviation from "unchanged": the source test.csv has a stray U+FEFF (byte order mark) glued to the head of
# its first row. Left in, it would create a bogus entity and a BOM in the middle of the output file, so it is removed.
BOM = '\ufeff'
stray = {s: [(int(i), c) for c in df.columns for i in df.index[df[c].str.contains(BOM, regex=False)]]
         for s, df in triples.items()}
print('stray U+FEFF characters in the source:', stray)
assert stray == {'train': [], 'valid': [], 'test': [(0, 'head')]}, stray
bad_name = triples['test'].at[0, 'head']
triples['test'].at[0, 'head'] = bad_name.replace(BOM, '')
print(f'repaired {bad_name!r} -> {triples["test"].at[0, "head"]!r}; '
      f'repaired name also occurs in train: {triples["test"].at[0, "head"] in set(triples["train"]["head"])}')

committed = read_triples_csv(DATA_DIR / COMMITTED_TRAIN)
assert committed.equals(triples['train']), 'downloaded train split differs from the committed train file'

for split, df in triples.items():
    write_tsv(df, OUT_DIR / f'{split}.tsv', header=False)

entities = sorted(set().union(*(set(df['head']) | set(df['tail']) for df in triples.values())))
pd.DataFrame({'triples': {s: len(df) for s, df in triples.items()},
              'self_loops': {s: int((df['head'] == df['tail']).sum()) for s, df in triples.items()},
              'duplicate_rows': {s: int(df.duplicated().sum()) for s, df in triples.items()}}).assign(
    entities_in_dataset=len(entities))

stray U+FEFF characters in the source: {'train': [], 'valid': [], 'test': [(0, 'head')]}


repaired '\ufeffStan_Collymore' -> 'Stan_Collymore'; repaired name also occurs in train: True


,triples,self_loops,duplicate_rows,entities_in_dataset
train,1079040,0,0,123182
valid,5000,0,0,123182
test,5000,0,0,123182


## 3. Class labels (raw)

`SimpleTypeFactsWordnetLevel.tsv` holds `<Entity>\trdf:type\t<wordnet_class_id>` rows. Angle brackets are stripped, so entities match the triple split and class IDs take the YAGO form `wordnet_person_100007846`.

In [4]:
with zipfile.ZipFile(downloads['SimpleTypeFactsWordnetLevel.zip']) as z:
    with z.open('SimpleTypeFactsWordnetLevel.tsv') as f:
        labels = pd.read_csv(f, sep='\t', header=None, names=['entity', 'predicate', 'class'],
                             quoting=csv.QUOTE_NONE, dtype='string', keep_default_na=False, encoding='utf-8')
assert len(labels) == 470_329 and set(labels['predicate']) == {'rdf:type'}
assert labels['entity'].str.match(r'^<.+>$').all()
labels = labels.assign(entity=labels['entity'].str[1:-1],
                       **{'class': labels['class'].str.removeprefix('<').str.removesuffix('>')})

# A few rows carry non-WordNet classes (yagoGeoEntity, owl:Thing); they are not part of the WordNet hierarchy.
is_wordnet = labels['class'].str.fullmatch(r'wordnet_.+_1\d{8}')
print('non-WordNet rows dropped:', labels.loc[~is_wordnet, 'class'].value_counts().to_dict())
labels = labels[is_wordnet].drop_duplicates().reset_index(drop=True)
print(f'{len(labels):,} (entity, class) rows, {labels["entity"].nunique():,} entities, {labels["class"].nunique():,} classes')

non-WordNet rows dropped: {'yagoGeoEntity': 6, 'owl:Thing': 2}


470,321 (entity, class) rows, 115,194 entities, 1,660 classes


### Entity-name coverage

Which dataset entities have at least one class in the label file? A low number would point at a Unicode or escaping mismatch, so the unmatched names are inspected.

In [5]:
import unicodedata

dataset_entities = set(entities)
labelled = set(labels['entity'])
unmatched = sorted(dataset_entities - labelled)
labelled_not_in_dataset = sorted(labelled - dataset_entities)

nfc_labelled = {unicodedata.normalize('NFC', e) for e in labelled}
nfc_rescued = sum(unicodedata.normalize('NFC', e) in nfc_labelled for e in unmatched)
unmatched_with_ws = sum(e != e.strip() for e in unmatched)
print(f'dataset entities: {len(dataset_entities):,}; with >=1 raw class: {len(dataset_entities & labelled):,} '
      f'({len(dataset_entities & labelled) / len(dataset_entities):.1%})')
print(f'dataset entities without any class in the label file: {len(unmatched):,}')
print(f'  matched after NFC normalisation: {nfc_rescued}; with leading/trailing whitespace: {unmatched_with_ws}; '
      f'pure ASCII names: {sum(e.isascii() for e in unmatched):,}')
print(f'label-file entities that do not occur in the dataset: {len(labelled_not_in_dataset)}')
print('sample of unmatched:', unmatched[:5] + unmatched[3000:3005])
print('sample of label-only:', labelled_not_in_dataset[:5])

dataset entities: 123,182; with >=1 raw class: 115,194 (93.5%)
dataset entities without any class in the label file: 7,988
  matched after NFC normalisation: 0; with leading/trailing whitespace: 0; pure ASCII names: 6,705
label-file entities that do not occur in the dataset: 0
sample of unmatched: ['1.F.C._Saarbrücken', '1._FC_Cologne', '1._FC_Nuremberg', '112_(group)', '1814_campaign_in_France', 'Harlingen_(Netherlands)', 'Harold_Andrews_(footballer_born_1897)', 'Harold_Martin_(politician)', 'Harold_Pearson_(footballer_born_1908)', 'Harriet_Frank,_Jr.']
sample of label-only: []


The unmatched names are not an encoding problem: normalising Unicode rescues none of them, and most are plain ASCII names. They are simply entities for which the authors' label file has no WordNet class (the file lists 115,194 entities, the dataset has more). They stay in the triple split, have no row in `entity_classes.tsv`, and cannot enter any experiment, because every experiment entity needs at least one class.

## 4. WordNet ancestor closure

The raw label file is not transitively closed: `wordnet_enterprise_108056231` has no members and many football players lack `person`. A YAGO class ID `wordnet_<lemma>_1<8-digit offset>` corresponds to the WordNet 3.0 noun synset with that offset. Each entity gets all ancestors of its classes, following `hypernyms() + instance_hypernyms()`, and ancestors are mapped back to YAGO IDs by offset. Experiment classes are matched **by offset**, not by string.

In [6]:
import nltk
nltk.download('wordnet', quiet=True)
from nltk.corpus import wordnet as wn

assert wn.get_version() == '3.0', wn.get_version()


def offset_of(class_id):
    digits = class_id.rsplit('_', 1)[1]
    assert len(digits) == 9 and digits[0] == '1', class_id
    return int(digits[1:])


def synset_at(offset):
    return wn.synset_from_pos_and_offset('n', offset)


def parents(synset):
    return synset.hypernyms() + synset.instance_hypernyms()


raw_classes = pd.Series(sorted(labels['class'].unique()), name='class')
raw_class_offset = {c: offset_of(c) for c in raw_classes}
experiment_offset = {c: offset_of(c) for c in EXPERIMENT_CLASSES}

# ancestors (including the class itself) of every class that occurs in the labels or in an experiment
needed = set(raw_class_offset.values()) | set(experiment_offset.values())
ancestors = {off: {off} | {s.offset() for s in synset_at(off).closure(parents)} for off in sorted(needed)}
all_offsets = set().union(*ancestors.values())
print(f'{len(needed):,} classes expand to {len(all_offsets):,} distinct WordNet synsets')

# Class ID per offset: the experiment ID if given, else the ID used in the label file, else derived from the lemma.
class_id_of = {off: f'wordnet_{synset_at(off).lemmas()[0].name().lower()}_1{off:08d}' for off in all_offsets}
ids_per_offset = {}
for c, off in raw_class_offset.items():
    ids_per_offset.setdefault(off, []).append(c)
assert all(len(ids) == 1 for ids in ids_per_offset.values()), 'several label-file IDs share one WordNet offset'
class_id_of.update({off: ids[0] for off, ids in ids_per_offset.items()})
class_id_of.update({off: c for c, off in experiment_offset.items()})
derived_mismatch = [c for c in raw_classes if c != f"wordnet_{synset_at(raw_class_offset[c]).lemmas()[0].name().lower()}_1{raw_class_offset[c]:08d}"]
print(f'label-file IDs whose name differs from the first WordNet lemma of their offset: {len(derived_mismatch)}', derived_mismatch[:5])
for c, off in experiment_offset.items():
    assert off in all_offsets
print({c: synset_at(off).name() for c, off in list(experiment_offset.items())[:4]})

1,664 classes expand to 2,378 distinct WordNet synsets
label-file IDs whose name differs from the first WordNet lemma of their offset: 3 ['wordnet_noble_110271677', 'wordnet_polar_zone_108573258', 'wordnet_premier_109907196']
{'wordnet_artist_109812338': 'artist.n.01', 'wordnet_ballplayer_109835506': 'ballplayer.n.01', 'wordnet_bay_109215664': 'bay.n.01', 'wordnet_biographer_109855433': 'biographer.n.01'}


In [7]:
ancestor_table = pd.DataFrame([(off, anc) for off, ancs in ancestors.items() for anc in ancs],
                              columns=['offset', 'ancestor'])
dataset_labels = labels[labels['entity'].isin(dataset_entities)].copy()
dataset_labels['offset'] = dataset_labels['class'].map(raw_class_offset)
closed = (dataset_labels[['entity', 'offset']].merge(ancestor_table, on='offset')
          [['entity', 'ancestor']].drop_duplicates())
closed['class'] = closed['ancestor'].map(class_id_of)
assert closed['class'].notna().all()
entity_classes = closed[['entity', 'class']].drop_duplicates().sort_values(['entity', 'class']).reset_index(drop=True)
entity_classes = entity_classes.astype('string')
print(f'raw pairs for dataset entities: {len(dataset_labels):,}; after closure: {len(entity_classes):,} '
      f'({entity_classes["entity"].nunique():,} entities, {entity_classes["class"].nunique():,} classes)')

raw pairs for dataset entities: 470,321; after closure: 1,634,127 (115,194 entities, 2,378 classes)


## 5. Sanity checks of the closure

Per experiment class: members in the raw file versus after closure (dataset entities only). Closure can only add members. Reference points from the paper (Fig. 1): about 67k persons and about 35k football players.

In [8]:
raw_by_offset = dataset_labels.groupby('offset')['entity'].nunique()
closed_by_offset = closed.groupby('ancestor')['entity'].nunique()
rows = []
for experiment, classes in EXPERIMENTS.items():
    for c in classes:
        off = experiment_offset[c]
        rows.append({'experiment': experiment, 'class': c, 'raw': int(raw_by_offset.get(off, 0)),
                     'closed': int(closed_by_offset.get(off, 0))})
class_counts = pd.DataFrame(rows).assign(added=lambda d: d['closed'] - d['raw'])
display(class_counts)

assert (class_counts['closed'] >= class_counts['raw']).all()
assert (class_counts['closed'] > 0).all(), 'every experiment class should have members'
focus = class_counts.drop_duplicates('class').set_index('class').loc[
    ['wordnet_person_100007846', 'wordnet_football_player_110101634', 'wordnet_enterprise_108056231'], ['raw', 'closed']]
display(focus)

,experiment,class,raw,closed,added
0,Level-1,wordnet_person_100007846,67476,67989,513
1,Level-1,wordnet_organization_108008335,4589,12169,7580
2,Level-1,wordnet_body_of_water_109225146,214,259,45
3,Level-1,wordnet_product_104007894,0,6570,6570
4,Level-2-Organizations,wordnet_musical_organization_108246613,337,343,6
5,Level-2-Organizations,wordnet_party_108256968,653,653,0
6,Level-2-Organizations,wordnet_enterprise_108056231,0,328,328
7,Level-2-Organizations,wordnet_nongovernmental_organization_108009834,5,32,27
8,Level-2-Waterbodies,wordnet_stream_109448361,0,84,84
9,Level-2-Waterbodies,wordnet_lake_109328904,43,45,2


,raw,closed
class,,
wordnet_person_100007846,67476,67989
wordnet_football_player_110101634,34554,34712
wordnet_enterprise_108056231,0,328


In [9]:
n_closed = entity_classes['entity'].nunique()
coverage = pd.Series({
    'dataset entities': f'{len(dataset_entities):,}',
    'with >=1 raw class': f'{len(dataset_entities & labelled):,}',
    'with >=1 class after closure': f'{n_closed:,}',
    'coverage': f'{n_closed / len(dataset_entities):.1%}',
})
display(coverage)
assert entity_classes['entity'].nunique() == len(dataset_entities & labelled)

dataset entities                123,182
with >=1 raw class              115,194
with >=1 class after closure    115,194
coverage                          93.5%
dtype: str

## 6. Write `entity_classes.tsv`

Only entities that occur in the dataset; sorted by entity, then class; header `entity\tclass`.

In [10]:
write_tsv(entity_classes, OUT_DIR / 'entity_classes.tsv', header=True)
entity_classes.head()

,entity,class
0,$_(film),wordnet_abstraction_100002137
1,$_(film),wordnet_artifact_100021939
2,$_(film),wordnet_creation_103129123
3,$_(film),wordnet_entity_100001740
4,$_(film),wordnet_event_100029378


## 7. Entity split

For every experiment, as in the authors' `classification.ipynb`: candidates are all dataset entities sorted by name; classes with fewer than 40 members are dropped; entities with at least one remaining experiment class are kept; then `train_test_split(entities, test_size=0.2, random_state=37)`, not stratified. `entity_split.tsv` stores, per kept entity, its split and its remaining experiment classes (sorted, comma-separated).

In [11]:
members = {c: set(g['entity']) for c, g in entity_classes[entity_classes['class'].isin(EXPERIMENT_CLASSES)].groupby('class')}

split_rows, summary = [], []
for experiment, classes in EXPERIMENTS.items():
    counts = {c: len(members.get(c, ())) for c in classes}
    kept = [c for c in classes if counts[c] >= MIN_CLASS_SIZE]
    dropped = [c for c in classes if counts[c] < MIN_CLASS_SIZE]
    entity_to_classes = {}
    for c in kept:
        for e in members[c]:
            entity_to_classes.setdefault(e, []).append(c)
    candidates_sorted = sorted(e for e in entities if e in entity_to_classes)
    train_entities, test_entities = train_test_split(candidates_sorted, test_size=TEST_SIZE, random_state=RANDOM_STATE)
    for split, group in (('train', train_entities), ('test', test_entities)):
        for e in sorted(group):
            split_rows.append((experiment, e, split, ','.join(sorted(entity_to_classes[e]))))
    summary.append({'experiment': experiment, 'classes_kept': len(kept), 'classes_dropped': len(dropped),
                    'dropped': ', '.join(c.rsplit('_', 1)[0].removeprefix('wordnet_') for c in dropped),
                    'entities': len(candidates_sorted), 'train': len(train_entities), 'test': len(test_entities),
                    'class_counts': {c.rsplit('_', 1)[0].removeprefix('wordnet_'): counts[c] for c in classes}})

entity_split = pd.DataFrame(split_rows, columns=['experiment', 'entity', 'split', 'classes']).astype('string')
write_tsv(entity_split, OUT_DIR / 'entity_split.tsv', header=True)
summary = pd.DataFrame(summary).set_index('experiment')
entity_split.groupby(['experiment', 'split']).size().unstack()

split,test,train
experiment,,
Level-1,17398,69589
Level-2-Organizations,264,1055
Level-2-Persons,3303,13212
Level-2-Waterbodies,42,166
Level-3-Artists,97,385
Level-3-Players,7346,29383
Level-3-Scientists,471,1880
Level-3-Writers,1075,4299


## 8. Acceptance checks

Written before the processing cells above. They read the files back from disk and compare against literals taken from the spec and the paper, so they do not share code with the processing. They fail until sections 1-7 have produced the files.

In [12]:
# 8.1 downloads are the pinned bytes
for name, (_, expected) in SOURCES.items():
    assert sha256(EXTERNAL_DIR / name) == expected, name

# 8.2 files are UTF-8 with LF line endings, no BOM, newline-terminated
OUTPUT_FILES = [f'{s}.tsv' for s in SPLITS] + ['entity_classes.tsv', 'entity_split.tsv']
for name in OUTPUT_FILES:
    raw_bytes = (OUT_DIR / name).read_bytes()
    raw_bytes.decode('utf-8')
    assert b'\xef\xbb\xbf' not in raw_bytes, f'{name} contains a byte order mark / U+FEFF'
    assert b'\r' not in raw_bytes, f'{name} contains CR'
    assert raw_bytes.endswith(b'\n'), name
print('files are UTF-8, LF-terminated')

files are UTF-8, LF-terminated


In [13]:
# 8.3 triple split: counts, content equal to the source, train equal to the committed file
reread = {s: read_tsv(OUT_DIR / f'{s}.tsv', header=False) for s in SPLITS}
for split, expected_rows in {'train': 1_079_040, 'valid': 5_000, 'test': 5_000}.items():
    assert len(reread[split]) == expected_rows, split
    source = read_triples_csv(EXTERNAL_DIR / f'{split}.csv')
    source = source.apply(lambda col: col.str.replace('\ufeff', '', regex=False))  # the one documented repair
    assert (reread[split].to_numpy() == source.to_numpy()).all(), f'{split}.tsv differs from its source'
committed_train = read_triples_csv(DATA_DIR / COMMITTED_TRAIN)
assert (reread['train'].to_numpy() == committed_train.to_numpy()).all(), 'train.tsv != committed train file'
dataset_entities_check = set()
for df in reread.values():
    dataset_entities_check |= set(df.iloc[:, 0]) | set(df.iloc[:, 2])
print(f'triple split ok; {len(dataset_entities_check):,} entities in the dataset')

triple split ok; 123,182 entities in the dataset


In [14]:
# 8.4 entity_classes.tsv
ec = read_tsv(OUT_DIR / 'entity_classes.tsv')
assert list(ec.columns) == ['entity', 'class']
assert not ec.duplicated().any(), 'duplicate (entity, class) rows'
assert ec.sort_values(['entity', 'class']).reset_index(drop=True).equals(ec), 'entity_classes.tsv is not sorted'
assert set(ec['entity']) <= dataset_entities_check, 'entity_classes contains entities outside the dataset'
assert not ec['class'].str.contains('[<>]').any(), 'class IDs must not carry angle brackets'
assert ec['class'].str.fullmatch(r'wordnet_.+_1\d{8}').all()
ec_classes = {c: set(g['entity']) for c, g in ec.groupby('class')}
n_in_dataset = len(dataset_entities_check)
print(f'{len(ec):,} pairs, {ec["entity"].nunique():,} of {n_in_dataset:,} dataset entities have a class')

# closure sanity (paper Fig. 1: person ~67k, football player ~35k); thresholds are literals
assert 60_000 <= len(ec_classes['wordnet_person_100007846']) <= 75_000
assert 30_000 <= len(ec_classes['wordnet_football_player_110101634']) <= 40_000
for c in EXPERIMENT_CLASSES:
    assert len(ec_classes.get(c, ())) > 0, f'{c} has no members'
# closure property: every football player is a person, every enterprise is an organization
assert ec_classes['wordnet_football_player_110101634'] <= ec_classes['wordnet_person_100007846']
assert ec_classes['wordnet_enterprise_108056231'] <= ec_classes['wordnet_organization_108008335']

1,634,127 pairs, 115,194 of 123,182 dataset entities have a class


In [15]:
# 8.5 entity_split.tsv, per experiment
es = read_tsv(OUT_DIR / 'entity_split.tsv')
assert list(es.columns) == ['experiment', 'entity', 'split', 'classes']
assert set(es['experiment']) == set(EXPERIMENTS), set(es['experiment']) ^ set(EXPERIMENTS)
assert set(es['split']) <= {'train', 'test'}
assert es['classes'].str.len().gt(0).all(), 'every row needs at least one class'
for experiment, group in es.groupby('experiment'):
    classes = EXPERIMENTS[experiment]
    train = set(group.loc[group['split'] == 'train', 'entity'])
    test = set(group.loc[group['split'] == 'test', 'entity'])
    n = len(train) + len(test)
    assert not group['entity'].duplicated().any(), experiment
    assert not (train & test), f'{experiment}: train and test overlap'
    assert len(test) == math.ceil(0.2 * n), (experiment, len(test), n)
    # class sizes among all dataset entities, independent of the split
    sizes = {c: len(ec_classes.get(c, ())) for c in classes}
    kept = {c for c in classes if sizes[c] >= 40}
    assert kept, f'{experiment}: no class survives the minimum size'
    used = set(','.join(group['classes']).split(','))
    assert used == kept, (experiment, used ^ kept)
    # entity set = all entities with at least one kept class
    expected_entities = set().union(*(ec_classes[c] for c in kept))
    assert train | test == expected_entities, experiment
    # per-row classes are sorted, and equal the entity's kept classes
    for e, cs in zip(group['entity'], group['classes']):
        parts = cs.split(',')
        assert parts == sorted(parts) and set(parts) <= kept
    sample = group.sample(min(2000, len(group)), random_state=0)
    for e, cs in zip(sample['entity'], sample['classes']):
        assert set(cs.split(',')) == {c for c in kept if e in ec_classes[c]}, (experiment, e)
    # every kept class has >= 40 members within the entity split
    per_class = pd.Series([c for cs in group['classes'] for c in cs.split(',')]).value_counts()
    assert (per_class >= 40).all()
print('entity split ok for', len(EXPERIMENTS), 'experiments')

entity split ok for 8 experiments


## 9. Summary

Per experiment: classes kept and dropped (fewer than 40 members), entities, entity-split sizes and members per class after the WordNet closure.

In [16]:
display(summary)
display(class_counts.set_index(['experiment', 'class'])[['raw', 'closed', 'added']])

,classes_kept,classes_dropped,dropped,entities,train,test,class_counts
experiment,,,,,,,
Level-1,4,0,,86987,69589,17398,"{'person': 67989, 'organization': 12169, 'body_of_water': 259, 'product': 6570}"
Level-2-Organizations,3,1,nongovernmental_organization,1319,1055,264,"{'musical_organization': 343, 'party': 653, 'enterprise': 328, 'nongovernmental_organization': 32}"
Level-2-Waterbodies,4,1,ocean,208,166,42,"{'stream': 84, 'lake': 45, 'ocean': 5, 'bay': 51, 'sea': 47}"
Level-2-Persons,5,0,,16515,13212,3303,"{'artist': 6523, 'officeholder': 1893, 'writer': 7150, 'scientist': 2662, 'politician': 2923}"
Level-3-Writers,7,0,,5374,4299,1075,"{'journalist': 698, 'poet': 712, 'novelist': 1191, 'scriptwriter': 3362, 'dramatist': 647, 'essayist': 333, 'biograp..."
Level-3-Scientists,10,0,,2351,1880,471,"{'social_scientist': 544, 'biologist': 346, 'physicist': 560, 'mathematician': 659, 'chemist': 334, 'linguist': 70, ..."
Level-3-Players,5,0,,36729,29383,7346,"{'football_player': 34712, 'ballplayer': 50, 'soccer_player': 6129, 'volleyball_player': 121, 'golfer': 248}"
Level-3-Artists,5,0,,482,385,97,"{'painter': 191, 'sculptor': 76, 'photographer': 202, 'illustrator': 49, 'printmaker': 54}"


raw  closed  added
experiment            class                                                               
Level-1               wordnet_person_100007846                        67476   67989    513
                      wordnet_organization_108008335                   4589   12169   7580
                      wordnet_body_of_water_109225146                   214     259     45
                      wordnet_product_104007894                           0    6570   6570
Level-2-Organizations wordnet_musical_organization_108246613            337     343      6
                      wordnet_party_108256968                           653     653      0
                      wordnet_enterprise_108056231                        0     328    328
                      wordnet_nongovernmental_organization_108009834      5      32     27
Level-2-Waterbodies   wordnet_stream_109448361                            0      84     84
                      wordnet_lake_109328904                             43      45      2
                      wordnet_ocean_109376198                             5       5      0
                      wordnet_bay_109215664                              51      51      0
                      wordnet_sea_109426788                              47      47      0
Level-2-Persons       wordnet_artist_109812338                         5352    6523   1171
                      wordnet_officeholder_110371450                   1893    1893      0
                      wordnet_writer_110794014                         5168    7150   1982
                      wordnet_scientist_110560637                      2146    2662    516
                      wordnet_politician_110450303                     2779    2923    144
Level-3-Writers       wordnet_journalist_110224578                      628     698     70
                      wordnet_poet_110444194                            711     712      1
                      wordnet_novelist_110363573                       1191    1191      0
                      wordnet_scriptwriter_110564905                      0    3362   3362
                      wordnet_dramatist_110030277                       647     647      0
                      wordnet_essayist_110064405                        333     333      0
                      wordnet_biographer_109855433                       75     304    229
Level-3-Scientists    wordnet_social_scientist_110619642                161     544    383
                      wordnet_biologist_109855630                       243     346    103
                      wordnet_physicist_110428004                       508     560     52
                      wordnet_mathematician_110301261                   632     659     27
                      wordnet_chemist_109913824                         315     334     19
                      wordnet_linguist_110264437                         65      70      5
                      wordnet_psychologist_110488865                    128     131      3
                      wordnet_geologist_110127689                        47      47      0
                      wordnet_computer_scientist_109951070              179     179      0
                      wordnet_research_worker_110523076                 189     189      0
Level-3-Players       wordnet_football_player_110101634               34554   34712    158
                      wordnet_ballplayer_109835506                       42      50      8
                      wordnet_soccer_player_110618342                  2873    6129   3256
                      wordnet_volleyball_player_110759047               121     121      0
                      wordnet_golfer_110136959                           13     248    235
Level-3-Artists       wordnet_painter_110391653                         179     191     12
                      wordnet_sculptor_110566072                         76      76      0
                      wordnet_photographer_110426749                